# Mapping Climate Crises with Python
### MLH Global Hack Week — Starter Notebook

In this workshop we'll take the **EM-DAT International Disaster Database** — a real, messy, 16,000+ row dataset — and turn it into an interactive map of climate-related natural disasters.

**Plan:**
1. Load and explore the raw data
2. Clean and filter it down to what we need
3. Explore patterns with pandas
4. Build an interactive map with Folium
5. Mini challenge — make it your own!

Follow along live — code cells marked `# TODO` are where we'll write code together.


## 0. Setup

Get the dataset yourself from [public.emdat.be/data](https://public.emdat.be/data) (free account required). Filter to 2000–2026, leave Classification/Countries blank, leave historical toggle off. Download as `.xlsx` and place it in the `data/` folder.

In [41]:
!pip install pandas folium openpyxl -q



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\Alex\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [42]:
import pandas as pd
import folium

# TODO: load the xlsx file with pd.read_excel
df = pd.read_excel("../data/emdat_db.xlsx")
print(df.shape)
df.head()

(16834, 47)


,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,GADM Admin Units,Entry Date,Last Update
0,2025-0303-ECU,No,nat-geo-ear-gro,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,ECU,...,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,2025-04-28,2026-05-15
1,2026-0153-KEN,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),NaN,NaN,KEN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-09,2026-05-15
2,2025-0750-AFG,No,nat-geo-ear-gro,Natural,Geophysical,Earthquake,Ground movement,GLIDE:EQ-2025-000153,NaN,AFG,...,NaN,NaN,NaN,180000.0,180000.0,100.0,NaN,NaN,2025-09-01,2026-05-15
3,2026-0305-AFG,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),NaN,NaN,AFG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-04,2026-05-11
4,2024-0962-AGO,No,nat-bio-epi-bac,Natural,Biological,Epidemic,Bacterial disease,GLIDE:EP-2025-000013,Cholera,AGO,...,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,2025-01-17,2026-05-15


## 1. Explore the mess

47 columns, 16,834 rows. Let's see what we're working with.

In [43]:
# TODO: use df.info() to see column types and null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16834 entries, 0 to 16833
Data columns (total 47 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   DisNo.                                     16834 non-null  str    
 1   Historic                                   16834 non-null  str    
 2   Classification Key                         16834 non-null  str    
 3   Disaster Group                             16834 non-null  str    
 4   Disaster Subgroup                          16834 non-null  str    
 5   Disaster Type                              16834 non-null  str    
 6   Disaster Subtype                           16834 non-null  str    
 7   External IDs                               4379 non-null   str    
 8   Event Name                                 5323 non-null   str    
 9   ISO                                        16834 non-null  str    
 10  Country                          

In [44]:
# TODO: check the 'Disaster Group' column with value_counts()
print(df['Disaster Group'].value_counts())

Disaster Group
Natural          10762
Technological     6072
Name: count, dtype: int64


We've got two groups: **Natural** disasters (floods, earthquakes, storms...) and **Technological** (industrial accidents, transport accidents). 

For "climate crises" we want the Natural ones.

In [45]:
# TODO: filter df to rows where 'Disaster Group' == 'Natural'
# Store the result as `natural`
natural = df[df["Disaster Group"] == "Natural"].copy()
print(f"Natural disasters: {len(natural)} rows")
natural['Disaster Type'].value_counts()

Natural disasters: 10762 rows


Disaster Type
Flood                          4271
Storm                          2877
Epidemic                        894
Earthquake                      698
Extreme temperature             568
Mass movement (wet)             500
Drought                         424
Wildfire                        344
Volcanic activity               133
Infestation                      29
Mass movement (dry)              14
Glacial lake outburst flood       8
Impact                            1
Animal incident                   1
Name: count, dtype: int64

## 2. The lat/long problem

To put events on a map we need coordinates. Let's check how many rows actually have them.

In [46]:
# TODO: count non-null values in 'Latitude' and compute the percentage of `natural` that has coordinates
percent = natural["Latitude"].notna().sum() / len(natural) * 100
print(f"{str(percent)}% have a coordinate")

17.22728117450288% have a coordinate


Discuss: what does it mean for our map if only some events have coordinates? Could this introduce bias?

In [47]:
# TODO: drop rows where Latitude or Longitude is missing
# Store the result as `geo`
geo = natural.dropna(subset=["Latitude", "Longitude"])
print(f"Remaining rows: {len(geo)}")
geo['Disaster Type'].value_counts()

Remaining rows: 1854


Disaster Type
Flood                  1006
Earthquake              670
Storm                   115
Volcanic activity        48
Mass movement (wet)      15
Name: count, dtype: int64

## 3. Select and clean the columns we need

We don't need all 47 columns. Let's pick the ones relevant to mapping and severity:
`Country`, `Disaster Type`, `Start Year`, `Latitude`, `Longitude`, `Total Deaths`, `Total Affected`

In [48]:
# TODO: select just these columns from `geo`
cols = ['Country', 'Disaster Type', 'Start Year', 'Latitude', 'Longitude', 'Total Deaths', 'Total Affected']
geo = geo[cols]
geo.head()

,Country,Disaster Type,Start Year,Latitude,Longitude,Total Deaths,Total Affected
0,Ecuador,Earthquake,2025,1.084,-79.532,NaN,8512.0
2,Afghanistan,Earthquake,2025,34.519,70.734,2200.0,1304600.0
70,Mexico,Earthquake,2026,16.875,-99.336,1.0,16152.0
124,Philippines,Flood,2010,6.780,125.300,27.0,40000.0
125,India,Flood,2010,26.235,92.250,NaN,30000.0


In [49]:
# TODO: check for nulls in the remaining columns with .isna().sum()
geo.isna().sum()

Country             0
Disaster Type       0
Start Year          0
Latitude            0
Longitude           0
Total Deaths      476
Total Affected    101
dtype: int64

`Total Deaths` and `Total Affected` have some missing values. We'll treat missing as 0 for mapping purposes.

In [50]:
# TODO: fill nulls in 'Total Deaths' and 'Total Affected' with 0 using .fillna(0)
# Then drop any remaining rows missing 'Country' or 'Start Year' with .dropna(subset=[...])
geo["Total Deaths"] = geo["Total Deaths"].fillna(0)
geo["Total Affected"] = geo["Total Affected"].fillna(0)

## 4. Quick exploration with pandas

Now that it's clean, let's ask some questions of the data.

In [51]:
# TODO: which disaster types are most common? (value_counts)
geo["Disaster Type"].value_counts()

Disaster Type
Flood                  1006
Earthquake              670
Storm                   115
Volcanic activity        48
Mass movement (wet)      15
Name: count, dtype: int64

In [52]:
# TODO: which countries have the most recorded events? (value_counts().head(10))
geo["Country"].value_counts()

Country
China                         166
Indonesia                     133
Philippines                    79
India                          77
Iran (Islamic Republic of)     62
                             ... 
Belgium                         1
Timor-Leste                     1
Ireland                         1
Israel                          1
Libya                           1
Name: count, Length: 155, dtype: int64

In [53]:
# TODO: filter to events from 2020 onwards and check how many remain
recent = geo[geo["Start Year"] >= 2020]
recent["Disaster Type"].value_counts()

Disaster Type
Earthquake    130
Flood          24
Storm           2
Name: count, dtype: int64

In [54]:
print("We had "+ str(len(geo))+" disasters from 2020 onwards")

We had 1854 disasters from 2020 onwards


## 5. Building the map with Folium

Let's start simple — a blank world map.

In [55]:
# TODO: create a folium.Map centered roughly on [20, 0] with zoom_start=2
m = folium.Map(location=[20,0], zoom_start=2)

Now let's add a single marker to check the syntax works.

In [ ]:
# TODO: add a folium.Marker for the first row of `geo`
# Use geo.iloc[0] to get the first row, and its Latitude/Longitude for location
m = folium.Map(location=[20, 0], zoom_start=2)

# your marker code here

m

ValueError: Custom tiles must have an attribution.

## 6. Plotting every event

Now let's loop through the dataframe and add a marker for every disaster. We'll:
- Use `CircleMarker` instead of `Marker` (lighter weight for many points)
- Colour-code by disaster type
- Size by `Total Affected` (so bigger circles = bigger impact)
- Add a popup with details

In [ ]:
# TODO: create a colour_map dictionary mapping each disaster type to a colour
# Folium accepts colour names like 'red', 'blue', 'green', 'purple', 'orange', etc.
disaster_types = geo['Disaster Type'].unique()
colours = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen', 'pink', 'gray', 'black', 'lightblue', 'darkpurple', 'beige']
colour_map = ...
colour_map

In [ ]:
import math

m = folium.Map(location=[20, 0], zoom_start=2)

for _, row in geo.iterrows():
    # TODO: compute a radius based on row['Total Affected']
    # Hint: try 3 + math.log10(affected + 1) * 2  to keep huge numbers from dominating
    radius = ...

    # TODO: add a folium.CircleMarker using:
    #   - location = [row['Latitude'], row['Longitude']]
    #   - radius = radius
    #   - color = colour_map.get(row['Disaster Type'], 'gray')
    #   - fill=True, fill_opacity=0.6
    #   - popup with disaster type, country, year, deaths, affected

m

## 7. Mini Challenge (10 minutes)

Pick one (or more!) of these to try:

1. **Filter to your region** — e.g. `geo[geo['Country'].str.contains('United Kingdom')]` and replot
2. **Filter to one disaster type** — e.g. just floods or just earthquakes, and see how the pattern changes
3. **Change the colour scheme** — maybe colour by *decade* instead of disaster type
4. **Add a legend** — Folium supports custom HTML legends, can you add one showing what each colour means?
5. **Save your map** — `m.save('my_climate_map.html')` and open it in a browser

Use the code cell below to experiment!

In [ ]:
# Your code here!


## 8. Wrap-up

You've taken a real, messy 16,000-row government dataset and turned it into an interactive map — handling missing data, filtering, and visualisation along the way.

**Where to go next:**
- Try the [GDIS dataset](https://www.nature.com/articles/s41597-021-00846-6) for subnational geocoding
- Explore `folium.plugins` — heatmaps, marker clusters, time-based animations (`TimestampedGeoJson`)
- Combine with [Our World in Data](https://ourworldindata.org/) climate datasets for richer context

Code for this workshop: **[github repo link]**

Thanks for joining!